# Lesson 2 Demo Notebook: Stateless and Stateful Agents

This notebook picks up where Lesson 1 left off, then tackles the central challenge of conversational AI: **the API is stateless, but users expect memory.**

1. **Chat Completions vs Responses API** — two API styles, same models, different tradeoffs
2. **The difficulty of testing** — non-deterministic outputs break traditional testing assumptions
3. **Stateless call** — each API call is independent; the model literally forgets
4. **Stateful call** — we fix this by replaying the full message history every turn
5. **Token growth** — this fix has quadratic cost: every turn re-sends all prior turns
6. **Sliding window + summarization** — bound the cost by keeping recent turns verbatim and compressing older ones into a memory summary
7. **Long-term memory** — persist facts across sessions by asking the LLM to extract what matters and saving to disk

Run cells in order. Sections 1–2 bridge from L1. Sections 3–7 build the memory story.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env")

client = OpenAI()
MODEL = "gpt-4.1-mini"
print("Client ready")


## 1) Chat Completions vs Responses API

In Lesson 1 we used the **Chat Completions API** (`client.chat.completions.create`). It's stateless: every call is a fresh start. The model only sees what you send in that request.

The **Responses API** (`client.responses.create`) can manage state **server-side** — if you chain calls with `previous_response_id`, the model remembers prior turns without you resending them.

| | Chat Completions | Responses API |
|---|---|---|
| **Mental model** | Single turn: messages in, message out | Multi-step: the model can take *actions* before responding |
| **Built-in tools** | None — you implement tool calls yourself | `web_search`, `file_search`, `code_interpreter` built in |
| **State** | You manage conversation history | Server can manage it (with `previous_response_id`) |
| **Context management** | You truncate/summarize manually | Server-side compaction available |

### Demo: no memory vs free memory

We'll ask the model a name, then ask it back — first with Chat Completions (no memory), then with Responses API (memory for free).

In [ ]:
# --- Chat Completions: NO memory ---
print("=== Chat Completions API (stateless) ===\n")

print('User: "My name is Alice."')
r1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Alice."}],
)
print(f"Assistant: {r1.choices[0].message.content}\n")

print('User: "What is my name?"')
r2 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is my name?"}],
)
print(f"Assistant: {r2.choices[0].message.content}")
print("  ^ It has no idea — each call is independent.\n")

# --- Responses API: memory for free ---
print("=== Responses API (server-side state) ===\n")

resp1 = client.responses.create(
    model=MODEL,
    input="My name is Alice.",
)
print("Turn 1:", resp1.output_text)

resp2 = client.responses.create(
    model=MODEL,
    input="What is my name?",
    previous_response_id=resp1.id,  # <-- this is all it takes
)
print("Turn 2:", resp2.output_text)
print("  ^ It remembers — the server kept the context.")

### Built-in tools: web search

The Responses API also has **built-in tools** — the model can autonomously search the web, run code, or read files *within a single API call*, before returning a final answer.

Here we give it `web_search`. The model decides to search, reads the results, and synthesizes an answer — all in one call. No tool-calling code on our side.

In [ ]:
try:
    resp = client.responses.create(
        model=MODEL,
        tools=[{"type": "web_search"}],
        tool_choice="auto",
        input="Find one positive AI news story from this week. Include source.",
    )
    print(resp.output_text)
except Exception as e:
    print("Web search demo unavailable in this project or region.")
    print(type(e).__name__, e)

### System prompts

The `system` role lets you steer *how* the model responds — its tone, format, persona — without changing the user's question. Same input, different behavior.

In [ ]:
question = "What is Software 3.0?"

system_prompts = [
    None,  # no system message
    "You are a sarcastic comedian. Keep it short.",
    "You are a formal academic. Respond in exactly one sentence.",
    "You are a 5-year-old explaining things to another 5-year-old.",
]

for sp in system_prompts:
    messages = []
    if sp:
        messages.append({"role": "system", "content": sp})
    messages.append({"role": "user", "content": question})

    r = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.7)

    label = f'"{sp}"' if sp else "(no system message)"
    print(f"System: {label}")
    print(f"  → {r.choices[0].message.content}\n")

## Temperature

Temperature controls **randomness** in the model's output. At `0` the model always picks the most likely next token. At higher values it samples more freely, producing more varied — and sometimes more creative — responses.

**Surprise:** even at `temperature=0`, you may see slightly different outputs across runs. This is because GPU floating-point arithmetic isn't perfectly deterministic — different hardware, different request routing, tiny rounding differences. OpenAI offers a `seed` parameter to improve reproducibility, but doesn't guarantee it.

Same prompt, three temperatures, three runs each.

In [ ]:
prompt = "Explain Software 3.0 in one sentence."

print("=== temperature=0 (no seed) ===")
for run in range(1, 6):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )
    print(f"  Run {run}: {r.choices[0].message.content}")

print("\n=== temperature=0, seed=42 ===")
for run in range(1, 6):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        seed=42,
    )
    print(f"  Run {run}: {r.choices[0].message.content}")

print("\n=== temperature=0.7 ===")
for run in range(1, 4):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    print(f"  Run {run}: {r.choices[0].message.content}")

print("\n=== temperature=1.5 ===")
for run in range(1, 4):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=1.5,
    )
    print(f"  Run {run}: {r.choices[0].message.content}")

## Structured Data Extraction

LLMs can extract structured data from unstructured text — but getting **valid, parseable output** is harder than it looks. We ask the model to return JSON with named entities. Then we compare:

1. A strong model (`gpt-5.2`) — reliably follows the format
2. A weak model (`gpt-4.1-nano`, `temperature=1`) — may produce broken JSON, miss entities, or hallucinate fields

This is a preview of a recurring theme: **the gap between "usually works" and "always works" is where engineering lives.**

In [ ]:
import json

NER_PROMPT = """Extract all named entities from the following text and return them as JSON.

Classify each entity as one of: PERSON, ORGANIZATION, LOCATION, DATE, or MISC.

Text: "Apple CEO Tim Cook announced a new partnership with Microsoft in Seattle last Tuesday. The deal was brokered by Goldman Sachs."

Return only valid JSON in this format:
{
  "entities": [
    {"text": "...", "type": "...", "start": <char_offset>}
  ]
}"""

# --- Strong model, temp=0 ---
print("=== gpt-5.2 (strong model, temperature=0) ===\n")
r_strong = client.chat.completions.create(
    model="gpt-5.2",
    messages=[{"role": "user", "content": NER_PROMPT}],
    temperature=0,
)
raw_strong = r_strong.choices[0].message.content
print(raw_strong)

try:
    parsed = json.loads(raw_strong)
    print(f"\n✓ Valid JSON — {len(parsed['entities'])} entities extracted")
except json.JSONDecodeError as e:
    print(f"\n✗ Invalid JSON: {e}")

# --- Strong model, temp=1 ---
print("\n\n=== gpt-5.2 (strong model, temperature=1) — 3 runs ===")
for run in range(1, 4):
    r = client.chat.completions.create(
        model="gpt-5.2",
        messages=[{"role": "user", "content": NER_PROMPT}],
        temperature=1,
    )
    raw = r.choices[0].message.content
    print(f"\n--- Run {run} ---")
    print(raw[:500])
    try:
        parsed = json.loads(raw)
        print(f"✓ Valid JSON — {len(parsed['entities'])} entities")
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON: {e}")

# --- Weak model, temp=1 ---
print("\n\n=== gpt-4.1-nano (weak model, temperature=1) — 3 runs ===")
for run in range(1, 4):
    r_weak = client.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[{"role": "user", "content": NER_PROMPT}],
        temperature=1,
    )
    raw_weak = r_weak.choices[0].message.content
    print(f"\n--- Run {run} ---")
    print(raw_weak[:500])
    try:
        parsed = json.loads(raw_weak)
        print(f"✓ Valid JSON — {len(parsed['entities'])} entities")
    except json.JSONDecodeError as e:
        print(f"✗ Invalid JSON: {e}")

## Few-Shot Learning

You can teach the model a new task **by example** — no fine-tuning needed. Just include a few input/output pairs in the prompt, and the model picks up the pattern.

This is called **few-shot prompting** (vs. **zero-shot**, where you give no examples). It's especially useful for tasks where the desired format or logic is hard to describe in words but easy to show.

Below we compare zero-shot vs few-shot on a sentiment classification task.

In [ ]:
test_review = "The battery lasts forever but the screen is dim and washes out in sunlight."

# --- Zero-shot: just ask ---
print("=== Zero-shot ===\n")
r_zero = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": f"Classify the sentiment of this product review as POSITIVE, NEGATIVE, or MIXED. Return only the label.\n\nReview: \"{test_review}\""},
    ],
    temperature=0,
)
print(f"Review: \"{test_review}\"")
print(f"Model:  {r_zero.choices[0].message.content}\n")

# --- Few-shot: teach by example ---
print("=== Few-shot (3 examples) ===\n")
r_few = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Classify product review sentiment. Reply with exactly one label: POSITIVE, NEGATIVE, or MIXED."},
        {"role": "user", "content": "Review: \"Absolutely love this phone, best purchase ever!\""},
        {"role": "assistant", "content": "POSITIVE"},
        {"role": "user", "content": "Review: \"Broke after two days. Total waste of money.\""},
        {"role": "assistant", "content": "NEGATIVE"},
        {"role": "user", "content": "Review: \"Camera is amazing but the software is buggy and slow.\""},
        {"role": "assistant", "content": "MIXED"},
        {"role": "user", "content": f"Review: \"{test_review}\""},
    ],
    temperature=0,
)
print(f"Review: \"{test_review}\"")
print(f"Model:  {r_few.choices[0].message.content}")
print("\nNotice how the few-shot examples teach the model the exact output format we want.")

## Chain-of-Thought Prompting

Some problems are hard to get right in one shot — especially reasoning, math, and multi-step logic. **Chain-of-thought (CoT)** prompting asks the model to show its reasoning step by step before giving a final answer.

Why does this help? The model's "thinking" tokens act as intermediate computation — each token can attend to the reasoning so far, reducing the chance of skipping a step.

Below we compare a direct answer vs chain-of-thought on a tricky word problem.

In [ ]:
problem = """A farmer has 15 sheep. All but 8 die. How many sheep are left?"""

# --- Direct answer ---
print("=== Direct (no CoT) ===\n")
r_direct = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": problem + "\n\nAnswer with just the number."},
    ],
    temperature=0,
)
print(f"Problem: {problem.strip()}")
print(f"Answer:  {r_direct.choices[0].message.content}\n")

# --- Chain-of-thought ---
print("=== Chain-of-Thought ===\n")
r_cot = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": problem + "\n\nThink step by step, then give your final answer."},
    ],
    temperature=0,
)
print(f"Problem: {problem.strip()}")
print(f"Answer:\n{r_cot.choices[0].message.content}")

# --- A harder problem where CoT matters more ---
hard_problem = """I have 3 boxes. Box A contains a cat. Box B contains a dog. Box C is empty.
I swap the contents of Box A and Box C, then swap the contents of Box B and Box A.
What is in each box now?"""

print("\n\n=== Harder problem — Direct ===\n")
r2_direct = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": hard_problem + "\n\nAnswer directly."},
    ],
    temperature=0,
)
print(f"Answer:\n{r2_direct.choices[0].message.content}")

print("\n=== Harder problem — Chain-of-Thought ===\n")
r2_cot = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": hard_problem + "\n\nThink step by step, then give your final answer."},
    ],
    temperature=0,
)
print(f"Answer:\n{r2_cot.choices[0].message.content}")

### Structured Chain-of-Thought

Instead of just saying "think step by step", you can **prescribe the output format** — forcing the model to separate reasoning from the final answer. This makes outputs easier to parse programmatically and reduces the chance of the model skipping steps.

In [ ]:
structured_cot_prompt = """Solve the following problem. 
First break the problem into reasoning steps, then compute the answer.

Output format:

Reasoning:
<step-by-step reasoning>

Answer:
<final answer>

Problem:

Roger has 5 tennis balls. 
He buys 2 more cans of tennis balls. 
Each can has 3 tennis balls.

How many tennis balls does he have now?"""

print("=== Structured CoT ===\n")
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": structured_cot_prompt}],
    temperature=0,
)
print(r.choices[0].message.content)

## Needle in a Haystack

Can the model find a single specific fact buried in a massive document?

We take the full text of *The Adventures of Sherlock Holmes* (~12,000 lines) with one small detail planted inside it, and ask the model to find it. This tests **long-context retrieval** — the ability to locate and extract a specific piece of information from a large input.

The "needle" is a modified line in the original text. Let's see if the model can find it.

In [ ]:
from pathlib import Path
import time

# Load the full Sherlock Holmes text (~12,000 lines, ~590 KB)
haystack_path = Path("labs/02_standalone_agents/prompting/sherlock.txt")
if not haystack_path.exists():
    haystack_path = Path("prompting/sherlock.txt")

haystack = haystack_path.read_text()
print(f"Loaded {len(haystack):,} characters ({len(haystack.splitlines()):,} lines)\n")

question = "Which building was depicted in the photo?"

start = time.time()
r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Answer the question based only on the provided text."},
        {"role": "user", "content": f"{haystack}\n\n---\nQuestion: {question}"},
    ],
    temperature=0,
)
elapsed = time.time() - start

print(f"Question: {question}")
print(f"Answer:   {r.choices[0].message.content}")
print(f"\nTokens — input: {r.usage.prompt_tokens:,}, output: {r.usage.completion_tokens}")
print(f"Elapsed: {elapsed:.2f}s")

## 2) The Difficulty of Testing

In traditional software, `sort([3,1,2])` must return `[1,2,3]`. One correct answer, deterministic, easy to assert.

LLM outputs are **non-deterministic by design**. The same prompt produces different (but equally valid) answers every time. No `assert ==` can capture "correct" when correct is a spectrum, not a point.

### What *can* we test?

- **Structural checks** — is the output non-empty? Is it within length bounds?
- **Property checks** — does a "one sentence" answer actually contain one sentence?
- **LLM-as-judge** — ask another LLM to grade the output (we'll cover this in depth later)

Run the cell below to see the same prompt produce three different answers.

In [ ]:
prompt = "Explain Software 3.0 in one sentence."

# Run the SAME prompt three times
for i in range(1, 4):
    r = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0.7,
    )
    answer = r.choices[0].message.content
    print(f"Run {i}: {answer}\n")

# All different, all correct. What would you assert?
assert answer is not None and len(answer) > 0, "Response should not be empty"
assert len(answer) < 500, "A one-sentence answer shouldn't be 500+ chars"
print("✓ Structural checks pass — but they don't tell us if the answer is *good*.")
print("  This is the gap we'll address with evaluation methods in later lessons.")

## 3) Stateless: each turn is independent

The Chat Completions API is **stateless by design** — every call is a fresh start. The model has no server-side session or hidden memory; it only sees the `messages` array you send in that request. Anything you don't include simply doesn't exist for the model.

Below we make two separate calls. Turn 1 tells the model a name; turn 2 asks for it — but since turn 2 doesn't include turn 1's messages, the model has no idea.

In [ ]:
turn1 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "My name is Alice."}]
)
print("Turn 1:", turn1.choices[0].message.content)

turn2 = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "What is my name?"}]
)
print("Turn 2:", turn2.choices[0].message.content)


## 4) Stateful: send history every turn

The fix is simple: **maintain a `messages` list and append every exchange to it.** On each new turn, send the entire list to the API. The model sees the full conversation and can answer follow-up questions.

This is exactly how ChatGPT works under the hood — the client sends the growing message history with every request. The "memory" lives in your code, not on OpenAI's servers.

In [ ]:
history = []

history.append({"role": "user", "content": "My name is Alice."})
r1 = client.chat.completions.create(model=MODEL, messages=history)
a1 = r1.choices[0].message.content
history.append({"role": "assistant", "content": a1})

history.append({"role": "user", "content": "What is my name?"})
r2 = client.chat.completions.create(model=MODEL, messages=history)
a2 = r2.choices[0].message.content

print("Assistant turn 1:", a1)
print("Assistant turn 2:", a2)
print("Prompt tokens on turn 2:", r2.usage.prompt_tokens)


## 5) Measure token growth

Sending full history works, but **the cost is quadratic**. Each turn re-sends every prior turn, so input tokens grow roughly as N × (average turn length). For a conversation averaging ~150 tokens per turn, the total input tokens over N turns is approximately **150 × N²/2**.

An 8-turn conversation doesn't feel expensive, but by turn 50 you're sending ~190k input tokens per call. This is the motivation for everything that follows: we need strategies to keep the context bounded while preserving as much information as possible.

In [ ]:
prompts = [
    "My name is Alice and I live in Trento.",
    "I work in computer science.",
    "I prefer concise answers.",
    "I am teaching an AI systems class.",
    "Summarize what you know about me so far.",
    "What details might be missing?",
    "Now give me three study tips.",
    "What is my name and where do I live?",
]

history = []
stats = []

for i, p in enumerate(prompts, start=1):
    history.append({"role": "user", "content": p})
    r = client.chat.completions.create(model=MODEL, messages=history, temperature=0.2)
    reply = r.choices[0].message.content
    history.append({"role": "assistant", "content": reply})

    stats.append((i, r.usage.prompt_tokens, r.usage.completion_tokens))

print("turn | prompt_tokens | completion_tokens")
for row in stats:
    print(f"{row[0]:>4} | {row[1]:>13} | {row[2]:>17}")


## 6) Sliding window + memory summarization

The solution is a two-part strategy from `3_agent_with_memory.py`:

1. **Sliding window** — keep the last N turns verbatim so the model has precise recent context
2. **Summarizer** — compress the older (dropped) turns into a short memory string using an LLM call

What gets sent to the API on each turn: `[system message with memory summary] + [recent window] + [new user message]`. The total token count stays roughly bounded regardless of conversation length.

The tradeoff: bounded cost at the price of **lossy compression**. The summary captures the gist but loses exact wording and small details. This is acceptable for most applications — humans do the same thing.

In [ ]:
from importlib.util import spec_from_file_location, module_from_spec
from pathlib import Path

# Import helpers from 3_agent_with_memory.py
mod_path = Path('labs/02_standalone_agents/3_agent_with_memory.py')
if not mod_path.exists():
    mod_path = Path('../labs/02_standalone_agents/3_agent_with_memory.py')

spec = spec_from_file_location('l2_memory', mod_path)
mod = module_from_spec(spec)
spec.loader.exec_module(mod)

# --- A realistic 10-turn conversation (hardcoded) ---
conversation = [
    {"role": "user", "content": "Hi! I'm studying computer science at UniTN."},
    {"role": "assistant", "content": "Welcome! That's a great program. How can I help?"},
    {"role": "user", "content": "I'm working on a project about recommendation systems."},
    {"role": "assistant", "content": "Nice topic! Are you focusing on collaborative filtering or content-based?"},
    {"role": "user", "content": "Collaborative filtering, specifically matrix factorization."},
    {"role": "assistant", "content": "Classic approach. SVD-based methods are a good starting point."},
    {"role": "user", "content": "Yes, I'm implementing SVD with PyTorch."},
    {"role": "assistant", "content": "Good choice for GPU acceleration. Are you using the MovieLens dataset?"},
    {"role": "user", "content": "Exactly, MovieLens 1M. Getting decent RMSE so far."},
    {"role": "assistant", "content": "That's a solid benchmark. What RMSE are you targeting?"},
]

# Step 1: Sliding window — keep only the last 2 turns
WINDOW = 2
window = mod.trim_to_window(conversation, window_turns=WINDOW)
older = conversation[:len(conversation) - len(window)]

print(f"Full conversation: {len(conversation)} messages ({len(conversation)//2} turns)")
print(f"Window keeps last {WINDOW} turns: {len(window)} messages")
print(f"Dropped (older): {len(older)} messages\n")

# Step 2: Summarize the dropped turns (this makes an API call)
print("--- Summarizing older turns (API call) ---")
memory = mod.summarize_turns(older, existing_memory="")
print(f"Memory summary:\n{memory}\n")

# Step 3: Build the final message array
user_question = "Can you remind me what dataset I'm using and suggest a next step?"
messages = mod.build_messages(memory, window, user_question)
print(f"--- Final message array sent to API: {len(messages)} messages ---")
for m in messages:
    role = m["role"].upper()
    preview = m["content"][:120].replace("\n", " ")
    print(f"  [{role}] {preview}{'...' if len(m['content']) > 120 else ''}")

# Step 4: Make the actual API call with windowed + summarized context
print("\n--- Model response ---")
response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0.3)
print(response.choices[0].message.content)
print(f"\nPrompt tokens: {response.usage.prompt_tokens} (bounded, not growing with full history)")

## 7) Long-term memory: persist across sessions

Everything above is **short-term memory** — it exists only while the conversation is running. Close the window and it's gone.

**Long-term memory** persists across sessions by saving facts to disk (or a database). The key insight: we ask the LLM itself to decide what's worth remembering. Given a conversation transcript, it extracts structured facts (name, location, preferences, projects) and we write them to a JSON file keyed by user ID.

On the next session, we load the user's facts and inject them into the system prompt — the model "remembers" things from previous conversations it never actually saw.

This is the pattern behind ChatGPT's memory feature, and it requires **user identity** (you need to know *whose* facts to load).

In [ ]:
import tempfile

# Import helpers from 4_agent_with_long_term_memory.py
mod4_path = Path('labs/02_standalone_agents/4_agent_with_long_term_memory.py')
if not mod4_path.exists():
    mod4_path = Path('../labs/02_standalone_agents/4_agent_with_long_term_memory.py')

spec4 = spec_from_file_location('l2_long_term', mod4_path)
mod4 = module_from_spec(spec4)
spec4.loader.exec_module(mod4)

# Override MEMORY_FILE to a temp path (avoid leaving artifacts)
tmp_memory = Path(tempfile.mkdtemp()) / "demo_memories.json"
mod4.MEMORY_FILE = tmp_memory

# --- Simulate a conversation where the user shares personal facts ---
demo_conversation = [
    {"role": "user", "content": "Hi, my name is Marco and I'm a PhD student in Trento."},
    {"role": "assistant", "content": "Nice to meet you, Marco! What are you working on?"},
    {"role": "user", "content": "I'm researching graph neural networks for drug discovery."},
    {"role": "assistant", "content": "Fascinating area! Are you using any specific GNN frameworks?"},
    {"role": "user", "content": "Mostly PyTorch Geometric. I also prefer concise explanations over long ones."},
    {"role": "assistant", "content": "Noted! I'll keep things brief. How's the research going?"},
]

# Step 1: Extract long-term facts (API call — the LLM decides what to persist)
print("--- Extracting long-term facts (API call) ---")
existing_memory = {"facts": [], "preferences": []}
extracted = mod4.extract_long_term_facts(demo_conversation, existing_memory)
print(f"Facts:       {extracted.get('facts', [])}")
print(f"Preferences: {extracted.get('preferences', [])}")

# Step 2: Save to disk, then load back (round-trip)
user_id = "marco"
mod4.save_user_memory(user_id, extracted)
loaded = mod4.get_user_memory(user_id)
print(f"\nSaved and reloaded for '{user_id}': {loaded}")

# Clean up temp file
tmp_memory.unlink(missing_ok=True)
tmp_memory.parent.rmdir()
print(f"\n(Temp file cleaned up — no artifacts left on disk)")

## Optional: run full CLI scripts

For the full interactive experience (recommended live in the terminal):
- `uv run python labs/02_standalone_agents/1_stateless_agent.py`
- `uv run python labs/02_standalone_agents/2_stateful_agent.py`
- `uv run python labs/02_standalone_agents/3_agent_with_memory.py`
- `uv run python labs/02_standalone_agents/4_agent_with_long_term_memory.py`